In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# P1-KN01: extract the selected UniDoc-Bench PDFs from the domain archives -> output becomes dataset P1-KD01
import hashlib
import tarfile
from pathlib import Path

from huggingface_hub import hf_hub_download

SELECTED = {"healthcare": ["3428789", "4671258"], "finance": ["6941304", "0458505"]}
OUTPUT = Path("/kaggle/working/unidoc_selected")
OUTPUT.mkdir(parents=True, exist_ok=True)

for domain, doc_ids in SELECTED.items():
    archive = hf_hub_download("Salesforce/UniDoc-Bench", f"{domain}_pdfs.tar.gz", repo_type="dataset", local_dir="/tmp/unidoc")
    with tarfile.open(archive, "r:gz") as tar:
        pdf_members = [member for member in tar.getmembers() if member.name.lower().endswith(".pdf")]
        print(domain, "| PDFs in archive:", len(pdf_members), "| example names:", [member.name for member in pdf_members[:3]])
        for doc_id in doc_ids:
            matches = [member for member in pdf_members if Path(member.name).stem == doc_id]
            if len(matches) != 1:
                print(f"  {doc_id}: {len(matches)} matches {[member.name for member in matches]} -> SKIPPED")
                continue
            content = tar.extractfile(matches[0]).read()
            (OUTPUT / f"unidoc_{domain}_{doc_id}.pdf").write_bytes(content)
            print(f"  {doc_id}: {matches[0].name} | {len(content):,} bytes | sha256 {hashlib.sha256(content).hexdigest()[:16]}")